In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB, CategoricalNB, ComplementNB, BernoulliNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier


In [3]:
# Combine all PDBs into a single dataframe
dfs = []
for filename in os.listdir('../data/features_ring'):
    dfs.append(pd.read_csv('../data/features_ring/' + filename, sep='\t'))
df = pd.concat(dfs)
df

,pdb_id,s_ch,s_resi,s_ins,s_resn,s_ss8,s_rsa,s_up,s_down,s_phi,...,t_down,t_phi,t_psi,t_ss3,t_a1,t_a2,t_a3,t_a4,t_a5,Interaction
0,7b5u,A,151,,K,E,0.361,11.0,9.0,-1.975,...,10.0,-1.061,-0.719,H,1.357,-1.453,1.477,0.113,-0.837,NaN
1,7b5u,A,152,,S,E,0.092,7.0,14.0,-1.428,...,12.0,-1.178,2.420,H,-0.228,1.399,-4.760,0.670,-2.647,NaN
2,7b5u,A,156,,L,H,0.030,23.0,4.0,-1.219,...,10.0,-1.472,-0.625,H,0.931,-0.179,-3.005,-0.503,-1.853,NaN
3,7b5u,A,159,,W,H,0.348,17.0,16.0,-0.993,...,22.0,1.340,3.101,L,-0.384,1.652,1.330,1.045,2.064,VDW
4,7b5u,A,197,,E,H,0.747,0.0,20.0,-1.057,...,19.0,-1.061,-0.698,H,-0.591,-1.302,-0.733,1.570,-0.146,HBOND
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253,2gom,A,137,,V,H,0.014,21.0,16.0,-1.046,...,15.0,-1.078,-0.739,H,1.831,-0.561,0.533,-0.277,1.648,NaN
254,2gom,A,114,,A,H,0.000,22.0,14.0,-1.071,...,14.0,-1.259,-0.601,H,-1.019,-0.987,-1.505,1.266,-0.912,VDW
255,2gom,A,128,,S,H,0.546,3.0,16.0,-1.038,...,19.0,-1.108,-0.777,H,1.538,-0.055,1.502,0.440,2.897,NaN
256,2gom,B,150,,V,H,0.528,5.0,13.0,-1.192,...,11.0,-1.073,-0.744,H,1.538,-0.055,1.502,0.440,2.897,HBOND


In [4]:
# Remove all rows with NaN in at least one column
# including rows with missing class (they could be false negatives)
df.dropna(inplace=True)

# Define ground truth values
y = df['Interaction'].astype('category')
y

3        VDW
4      HBOND
9      HBOND
10       VDW
12     HBOND
       ...  
250    HBOND
251      VDW
252    HBOND
254      VDW
256    HBOND
Name: Interaction, Length: 1471380, dtype: category
Categories (7, object): ['HBOND', 'IONIC', 'PICATION', 'PIHBOND', 'PIPISTACK', 'SSBOND', 'VDW']

In [5]:
# Define training features
X = df[['s_rsa', 's_up', 's_down', 's_phi', 's_psi', 's_a1', 's_a2', 's_a3', 's_a4', 's_a5', 
        't_rsa', 't_up', 't_down', 't_phi', 't_psi', 't_a1', 't_a2', 't_a3', 't_a4', 't_a5']]

# Calculate percentiles and transform into categories
X = X.rank(pct=True).round(1).astype('category') 
X

,s_rsa,s_up,s_down,s_phi,s_psi,s_a1,s_a2,s_a3,s_a4,s_a5,t_rsa,t_up,t_down,t_phi,t_psi,t_a1,t_a2,t_a3,t_a4,t_a5
3,0.8,0.5,0.4,0.9,0.3,0.3,0.6,0.5,0.0,0.4,0.5,0.3,0.8,1.0,1.0,0.4,1.0,0.6,0.7,0.9
4,1.0,0.0,0.6,0.8,0.4,0.8,0.1,0.7,0.4,0.4,0.7,0.0,0.6,0.8,0.3,0.4,0.1,0.4,1.0,0.5
9,0.2,0.8,0.0,0.3,0.6,0.0,0.5,0.4,0.8,0.2,0.3,0.5,0.1,0.5,1.0,0.7,0.7,0.1,0.3,0.0
10,0.2,0.8,0.0,0.3,0.6,0.0,0.5,0.4,0.8,0.2,0.3,0.5,0.1,0.5,1.0,0.7,0.7,0.1,0.3,0.0
12,0.6,0.8,0.9,0.2,0.7,0.5,0.7,0.9,0.7,0.8,0.3,0.6,0.8,0.2,0.9,0.0,0.5,0.4,0.8,0.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
250,0.5,0.7,0.1,0.8,0.2,0.3,0.1,0.4,1.0,0.5,0.9,0.4,0.1,0.4,0.5,0.2,0.2,0.3,0.9,0.3
251,0.7,0.1,0.6,0.3,0.9,0.5,0.7,0.9,0.7,0.8,0.9,0.1,0.8,0.2,0.9,0.0,0.5,0.4,0.8,0.2
252,0.5,0.2,0.6,0.9,0.2,0.0,0.5,0.4,0.8,0.2,0.9,0.1,0.9,0.1,0.9,0.0,0.5,0.4,0.8,0.2
254,0.1,0.8,0.3,0.7,0.2,0.3,0.1,0.4,1.0,0.5,0.6,0.5,0.3,0.5,0.4,0.2,0.2,0.3,0.9,0.3


In [6]:
# Split the dataset to define training and testing examples
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=0)

### Test different versions of Naive Bayes

In [7]:
nb = GaussianNB()
y_pred = nb.fit(X_train, y_train).predict(X_test)
print("Number of mislabeled points out of a total %d points : %d" % (X_test.shape[0], (y_test != y_pred).sum()))

Number of mislabeled points out of a total 147138 points : 72503


In [8]:
nb = MultinomialNB()
y_pred = nb.fit(X_train, y_train).predict(X_test)
print("Number of mislabeled points out of a total %d points : %d" % (X_test.shape[0], (y_test != y_pred).sum()))

Number of mislabeled points out of a total 147138 points : 65403


In [9]:
nb = ComplementNB()
y_pred = nb.fit(X_train, y_train).predict(X_test)
print("Number of mislabeled points out of a total %d points : %d" % (X_test.shape[0], (y_test != y_pred).sum()))

Number of mislabeled points out of a total 147138 points : 89813


In [10]:
nb = BernoulliNB()
y_pred = nb.fit(X_train, y_train).predict(X_test)
print("Number of mislabeled points out of a total %d points : %d" % (X_test.shape[0], (y_test != y_pred).sum()))

Number of mislabeled points out of a total 147138 points : 65186


In [12]:
nb = CategoricalNB()
y_pred = nb.fit(X_train, y_train).predict(X_test)
print("Number of mislabeled points out of a total %d points : %d" % (X_test.shape[0], (y_test != y_pred).sum()))

Number of mislabeled points out of a total 147138 points : 64542


In [11]:
qda = QuadraticDiscriminantAnalysis()
y_pred = qda.fit(X_train, y_train).predict(X_test)
print("Number of mislabeled points out of a total %d points : %d" % (X_test.shape[0], (y_test != y_pred).sum()))

/home/noob/.local/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:935: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")


Number of mislabeled points out of a total 147138 points : 69172


/home/noob/.local/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:935: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
/home/noob/.local/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:935: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
/home/noob/.local/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:935: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
/home/noob/.local/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:935: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
/home/noob/.local/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:935: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")


Accuracy: 0.53 (+/- 0.00)


In [18]:
random_forest = RandomForestClassifier(n_estimators=700, max_depth=3, random_state=0)
y_pred = random_forest.fit(X_train, y_train).predict(X_test)
print("Number of mislabeled points out of a total %d points : %d" % (X_test.shape[0], (y_test != y_pred).sum()))


Number of mislabeled points out of a total 147138 points : 64445


In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
def one_hot_encode(label):
    labels = {'HBOND'     :[1.0,0.0,0.0,0.0,0.0,0.0,0.0], 
              'IONIC'     :[0.0,1.0,0.0,0.0,0.0,0.0,0.0],
              'PICATION'  :[0.0,0.0,1.0,0.0,0.0,0.0,0.0], 
              'PIHBOND'   :[0.0,0.0,0.0,1.0,0.0,0.0,0.0], 
              'PIPISTACK' :[0.0,0.0,0.0,0.0,1.0,0.0,0.0], 
              'SSBOND'    :[0.0,0.0,0.0,0.0,0.0,1.0,0.0], 
              'VDW'       :[0.0,0.0,0.0,0.0,0.0,0.0,1.0]}
    return labels[label]

class DataFrameDataset(Dataset):
    def __init__(self, dataframe, is_label_set=False, transform=None):

        self.dataframe = dataframe
        self.is_label_set = is_label_set

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        
        sample = self.dataframe.iloc[idx]

        if self.is_label_set:
            return torch.tensor(one_hot_encode(sample))

        else:
        
            return torch.tensor(sample.to_numpy())

class SimpleMLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        """
        Args:
            input_size (int): Number of input features.
            hidden_size (int): Number of neurons in the hidden layer.
            num_classes (int): Number of output classes.
        """
        super(SimpleMLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)  # The raw scores (logits)
        return x

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleMLP(20, 36, 7).double()
model.to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0002)
batch_size = 64


X_train_ds, y_train_ds = DataLoader(DataFrameDataset(X_train), batch_size, shuffle=True), DataLoader(DataFrameDataset(y_train, is_label_set=True), batch_size, shuffle=True)

In [ ]:
model.train()
for epoch in range(20):  
    for i, batch in enumerate(zip(X_train_ds, y_train_ds)):
        X=batch[0].to(device)
        Y=batch[1].to(device)
        pred = model(X)
        loss = criterion(pred, Y)
        loss.backward()
        optimizer.step()

        if i%50 == 0:
            print(loss.item())
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/100], Loss: {loss.item():.4f}')